Libraries

In [90]:
import numpy as np
import os
import random
from PIL import Image, ImageEnhance, ImageFilter, ImageOps, ImageStat, ImageDraw
import cv2 # Wymagane dla transformacji perspektywy
import time # Do mierzenia czasu

Loading and preparing files

In [91]:
PANEL_DIR = "panele"
ROOF_DIR = "dachy"
OUTPUT_BASE_DIR = "aug4"
TOTAL_IMAGES_TO_GENERATE = 250
CLASS_ID = 0

FINAL_IMAGE_SIZE = (640, 640) # Końcowy rozmiar obrazu
MIN_PIXEL_SIZE = 10 # Minimalny rozmiar panelu

PERSPECTIVE_CHANCE = 0.9
PERSPECTIVE_STRENGTH = (0.05, 0.15) 
BLUR_STRENGTH = (4.0, 7.0) # Zwiększone rozmycie
ALPHA_BLUR_RANGE = (2, 5)  

MAX_PANELS_PER_IMAGE = 1
MIN_PANELS_PER_IMAGE = 1
MAX_PLACEMENT_ATTEMPTS = 100 
SCALE_RANGE = (0.10, 0.25) 
POSITION_MAX_Y = 0.80 


LOWER_RED_1 = np.array([0, 70, 50])
UPPER_RED_1 = np.array([10, 255, 255])
LOWER_RED_2 = np.array([160, 70, 50])
UPPER_RED_2 = np.array([180, 255, 255])
LOWER_ORANGE_BRICK = np.array([11, 70, 50])
UPPER_ORANGE_BRICK = np.array([25, 255, 255])

LOWER_NAVY = np.array([100, 70, 20])
UPPER_NAVY = np.array([130, 255, 100])
LOWER_BLACK = np.array([0, 0, 0])
UPPER_BLACK = np.array([180, 255, 40]) 

LOWER_DARK_GRAY = np.array([0, 0, 41]) 
UPPER_DARK_GRAY = np.array([180, 50, 100]) 

OUTPUT_IMG_DIR = os.path.join(OUTPUT_BASE_DIR, "obrazy")
OUTPUT_LABEL_DIR = os.path.join(OUTPUT_BASE_DIR, "etykiety")
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

print(f"Katalogi wyjściowe gotowe w: {OUTPUT_BASE_DIR}")
print(f"Końcowy rozmiar obrazu: {FINAL_IMAGE_SIZE}")
print(f"NOWA ZASADA: Dokładnie {MAX_PANELS_PER_IMAGE} panel na obraz.")
print(f"NOWA ZASADA: Rozmiar panelu = 1/5 powierzchni wykrytego dachu.")
print(f"NOWY Zakres rozmycia panelu: {BLUR_STRENGTH}")
print(f"Logika kolorów: P1(Czerwone/Pomarańcze) -> P2(Granat/Czerń) -> P3(Ciemnoszare)")

Katalogi wyjściowe gotowe w: aug4
Końcowy rozmiar obrazu: (640, 640)
NOWA ZASADA: Dokładnie 1 panel na obraz.
NOWA ZASADA: Rozmiar panelu = 1/5 powierzchni wykrytego dachu.
NOWY Zakres rozmycia panelu: (4.0, 7.0)
Logika kolorów: P1(Czerwone/Pomarańcze) -> P2(Granat/Czerń) -> P3(Ciemnoszare)


Preztwarzanie samego panela - modyfikacje

In [92]:
def preserve_alpha_apply_rgb(img, operation_func):
    if not isinstance(img, Image.Image): return img
    if img.mode != 'RGBA':
        try: return operation_func(img.convert('RGB'))
        except Exception as e: return img
    original_img = img.copy()
    try:
        alpha = img.getchannel('A')
        if ImageStat.Stat(alpha).sum[0] < 1: return original_img
        rgb = img.convert('RGB')
        modified_rgb = operation_func(rgb)
        if not isinstance(modified_rgb, Image.Image): return original_img
        modified_rgb.putalpha(alpha)
        return modified_rgb
    except Exception as e: return original_img

In [93]:
def apply_scaling(img, abs_boxes):
    original_img, original_boxes = img.copy(), list(abs_boxes)
    try:
        width, height = img.size
        if width <= 0 or height <= 0: return original_img, original_boxes
        scale = random.uniform(0.7, 1.3)
        new_w = max(1, int(width * scale))
        new_h = max(1, int(height * scale))
        try: resample_filter = Image.Resampling.LANCZOS
        except AttributeError: resample_filter = Image.LANCZOS
        img = img.resize((new_w, new_h), resample_filter)
        scale_x = img.size[0] / width if width > 0 else 0
        scale_y = img.size[1] / height if height > 0 else 0
        abs_boxes = [[cls, int(x1*scale_x), int(y1*scale_y), int(x2*scale_x), int(y2*scale_y)] for cls, x1, y1, x2, y2 in abs_boxes]
        return img, abs_boxes
    except Exception as e:
        # print(f"Warning: Error in apply_scaling ({e}). Returning original.")
        return original_img, original_boxes

In [94]:
def apply_rotation(img):
    original_img = img.copy()
    try:
        angle = random.uniform(-15, 15); img = img.convert("RGBA")
        try: img = img.rotate(angle, expand=True, resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0, 0))
        except AttributeError: img = img.rotate(angle, expand=True, resample=Image.BILINEAR, fillcolor=(0, 0, 0, 0))
        return img
    except Exception as e: return original_img

In [95]:
def apply_blur(img):
    if random.random() < 0.9: # 90% szans
        return preserve_alpha_apply_rgb(img, lambda rgb_img: rgb_img.filter(ImageFilter.GaussianBlur(random.uniform(BLUR_STRENGTH[0], BLUR_STRENGTH[1]))))
    return img

In [96]:
def apply_flip(img, abs_boxes):
    if random.random() < 0.5:
        original_img, original_boxes = img.copy(), list(abs_boxes)
        try:
            img = ImageOps.mirror(img.convert("RGBA")); w = img.size[0]
            abs_boxes = [[cls, w - x2, y1, w - x1, y2] for cls, x1, y1, x2, y2 in abs_boxes]
            return img, abs_boxes
        except Exception as e: return original_img, original_boxes
    return img, abs_boxes

In [97]:
def apply_noise(img):
    if random.random() < 0.5:
        original_img = img.copy()
        try:
            img = img.convert("RGBA"); np_img = np.array(img).astype(np.int16); alpha_channel = np_img[:, :, 3]
            rgb_channels = np_img[:, :, :3]; noise = np.random.normal(0, 15, rgb_channels.shape)
            rgb_channels = np.clip(rgb_channels + noise, 0, 255).astype(np.uint8)
            final_img_np = np.dstack((rgb_channels, alpha_channel)); img = Image.fromarray(final_img_np, 'RGBA')
            return img
        except Exception as e: return original_img
    return img

In [98]:
def apply_perspective_transform(img, abs_boxes):
    if random.random() < PERSPECTIVE_CHANCE:
        original_img, original_boxes = img.copy(), list(abs_boxes)
        try:
            img = img.convert("RGBA"); panel_w, panel_h = img.size
            if panel_w < 2 or panel_h < 2: return original_img, original_boxes
            src_points = np.float32([[0, 0], [panel_w, 0], [panel_w, panel_h], [0, panel_h]])
            max_shift = max(panel_w, panel_h) * random.uniform(PERSPECTIVE_STRENGTH[0], PERSPECTIVE_STRENGTH[1])
            small_shift = max(panel_w, panel_h) * 0.02
            dst_points = np.float32([ [random.uniform(0, max_shift), random.uniform(0, max_shift)], [panel_w - random.uniform(0, max_shift), random.uniform(0, max_shift)], [panel_w - random.uniform(0, small_shift), panel_h - random.uniform(0, small_shift)], [random.uniform(0, small_shift), panel_h - random.uniform(0, small_shift)] ])
            if np.linalg.det(np.array([[dst_points[0][0], dst_points[0][1], 1], [dst_points[1][0], dst_points[1][1], 1], [dst_points[2][0], dst_points[2][1], 1]])) < 1e-6: return original_img, original_boxes
            M = cv2.getPerspectiveTransform(src_points, dst_points)
            img_cv = np.array(img); warped_img_cv = cv2.warpPerspective(img_cv, M, (panel_w, panel_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
            warped_img = Image.fromarray(warped_img_cv, 'RGBA')
            try:
                alpha_check = warped_img.getchannel('A')
                if ImageStat.Stat(alpha_check).sum[0] < 10: return original_img, original_boxes
            except Exception as e_alpha_persp: return original_img, original_boxes
            bbox_pts = np.float32([[[0, 0]], [[panel_w, 0]], [[panel_w, panel_h]], [[0, panel_h]]])
            warped_bbox_pts = cv2.perspectiveTransform(bbox_pts, M)
            if warped_bbox_pts is None or np.isnan(warped_bbox_pts).any(): return original_img, original_boxes
            warped_bbox_pts = warped_bbox_pts.squeeze(); x_min, y_min = np.min(warped_bbox_pts, axis=0); x_max, y_max = np.max(warped_bbox_pts, axis=0)
            if x_max <= x_min or y_max <= y_min: return original_img, original_boxes
            new_abs_boxes = [[CLASS_ID, int(x_min), int(y_min), int(x_max), int(y_max)]]
            return warped_img, new_abs_boxes
        except Exception as e: return original_img, original_boxes
    return img, abs_boxes

In [99]:
def blur_alpha_channel(img):
    """Rozmywa tylko kanał alfa, aby zmiękczyć krawędzie."""
    if not isinstance(img, Image.Image) or img.mode != 'RGBA': return img
    original_img = img.copy()
    try:
        alpha = img.getchannel('A'); alpha_np = np.array(alpha)
        blur_radius = random.randint(ALPHA_BLUR_RANGE[0], ALPHA_BLUR_RANGE[1])
        kernel_size = (blur_radius * 2 + 1, blur_radius * 2 + 1)
        if kernel_size[0] <= 0 or kernel_size[1] <= 0: return original_img
        blurred_alpha_np = cv2.GaussianBlur(alpha_np, kernel_size, 0)
        blurred_alpha = Image.fromarray(blurred_alpha_np, mode='L')
        rgb = img.convert('RGB'); rgb.putalpha(blurred_alpha)
        return rgb
    except Exception as e: return original_img

In [100]:
def transform_image_and_boxes(img, abs_boxes):
    """Główny potok augmentacji."""
    img, abs_boxes = apply_scaling(img, abs_boxes)
    img = apply_rotation(img)
    img = apply_blur(img) # Rozmycie panelu (RGB)
    img = apply_noise(img)
    img, abs_boxes = apply_flip(img, abs_boxes)
    img, abs_boxes = apply_perspective_transform(img, abs_boxes)
    
    # NAPRAWA: Przeniesione rozmycie krawędzi (alpha) TUTAJ
    img = blur_alpha_channel(img) 
    
    # NAPRAWA: Ostateczna kontrola widoczności
    try:
        alpha_check = img.getchannel('A')
        if ImageStat.Stat(alpha_check).sum[0] < 50: 
            return None, [] # Zwróć None, jeśli panel jest niewidoczny
    except Exception as e_alpha:
        return None, [] 
        
    return img, abs_boxes

In [101]:
def create_priority_mask(hsv_img, priority_level):
    """
    Tworzy maskę binarną dla określonej grupy priorytetów kolorów.
    """
    mask = np.zeros(hsv_img.shape[:2], dtype=np.uint8)
    
    if priority_level == 1:
       
        mask_red1 = cv2.inRange(hsv_img, LOWER_RED_1, UPPER_RED_1)
        mask_red2 = cv2.inRange(hsv_img, LOWER_RED_2, UPPER_RED_2)
        mask_orange = cv2.inRange(hsv_img, LOWER_ORANGE_BRICK, UPPER_ORANGE_BRICK)
        mask = cv2.bitwise_or(mask_red1, mask_red2)
        mask = cv2.bitwise_or(mask, mask_orange)
    
    elif priority_level == 2:
        
        mask_navy = cv2.inRange(hsv_img, LOWER_NAVY, UPPER_NAVY)
        mask_black = cv2.inRange(hsv_img, LOWER_BLACK, UPPER_BLACK)
        mask = cv2.bitwise_or(mask_navy, mask_black)
    
    elif priority_level == 3:
        
        mask = cv2.inRange(hsv_img, LOWER_DARK_GRAY, UPPER_DARK_GRAY)
    
    
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=2)
    
    return mask

Panele na dachach

In [102]:
def blur_alpha_channel(img, blur_radius=2):
    """Blurs only the alpha channel of an RGBA image using OpenCV."""
    if not isinstance(img, Image.Image) or img.mode != 'RGBA': return img
    original_img = img.copy()
    try:
        alpha = img.getchannel('A')
        alpha_np = np.array(alpha)
        kernel_size = (blur_radius * 2 + 1, blur_radius * 2 + 1)
        # Ensure kernel size is valid
        if kernel_size[0] <= 0 or kernel_size[1] <= 0: return original_img
        blurred_alpha_np = cv2.GaussianBlur(alpha_np, kernel_size, 0)
        blurred_alpha = Image.fromarray(blurred_alpha_np, mode='L')
        rgb = img.convert('RGB')
        rgb.putalpha(blurred_alpha)
        return rgb
    except Exception as e:
        # print(f"Warning: Failed to blur alpha channel ({e}).")
        return original_img

In [103]:
def composite_panel_realistically(panel_img_aug, abs_boxes_aug, background_img):
    """Composites panel onto background, ensuring visibility."""
    try:
        # Check input panel validity
        if panel_img_aug is None or not hasattr(panel_img_aug, 'mode') or panel_img_aug.mode != 'RGBA': return None, None
        try:
            alpha_in_check = panel_img_aug.getchannel('A')
            if ImageStat.Stat(alpha_in_check).sum[0] < 1: return None, None
        except: return None, None # Error getting alpha

        bg_w, bg_h = background_img.size
        panel_w, panel_h = panel_img_aug.size
        if panel_w <=0 or panel_h <= 0: return None, None

        # Calculate target size and scale
        target_fill_percent = random.uniform(SCALE_RANGE[0], SCALE_RANGE[1])
        target_panel_w = int(bg_w * target_fill_percent)
        scale_needed = target_panel_w / panel_w if panel_w > 0 else 0
        if scale_needed <= 0: return None, None

        # Enforce minimum size
        new_w = max(MIN_PIXEL_SIZE, target_panel_w)
        new_h = max(MIN_PIXEL_SIZE, int(panel_h * scale_needed))

        # Resize panel
        try: resample_filter = Image.Resampling.LANCZOS
        except AttributeError: resample_filter = Image.LANCZOS
        panel_img_aug = panel_img_aug.resize((new_w, new_h), resample_filter)
        panel_w, panel_h = new_w, new_h # Update size

        # Scale BBox
        abs_boxes_scaled = []
        for cls, x1, y1, x2, y2 in abs_boxes_aug:
            x1_s, y1_s = int(x1 * scale_needed), int(y1 * scale_needed)
            x2_s, y2_s = int(x2 * scale_needed), int(y2 * scale_needed)
            if x2_s <= x1_s or y2_s <= y1_s: return None, None # Invalid box after scaling
            abs_boxes_scaled.append([cls, x1_s, y1_s, x2_s, y2_s])
        if not abs_boxes_scaled: return None, None # No valid boxes

        # Determine placement position
        max_x = bg_w - panel_w
        max_y = int(bg_h * POSITION_MAX_Y) - panel_h
        if max_x < 0 or max_y < 0: return None, None # Panel won't fit

        x_offset = random.randint(0, max_x)
        y_offset = random.randint(0, max_y)

        # (Color matching disabled)
        # background_patch = background_img.crop((x_offset, y_offset, x_offset + panel_w, y_offset + panel_h))
        # panel_img_aug = match_color_and_lighting(panel_img_aug, background_patch)

        # Blur alpha channel edges
        panel_img_aug = blur_alpha_channel(panel_img_aug, blur_radius=random.randint(1, 3))

        # --- FINAL ALPHA CHECK REMOVED ---
        # We assume if it got here, it's visible enough to paste.

        # Composite image
        composite_img = background_img.copy()
        composite_img.paste(panel_img_aug, (x_offset, y_offset), panel_img_aug)

        # Calculate final BBox coordinates
        final_abs_boxes = []
        for cls, x1, y1, x2, y2 in abs_boxes_scaled:
            fx1, fy1 = x1 + x_offset, y1 + y_offset
            fx2, fy2 = x2 + x_offset, y2 + y_offset
            if fx2 <= fx1 or fy2 <= fy1: return None, None # Final check after offset
            final_abs_boxes.append([cls, fx1, fy1, fx2, fy2])
        if not final_abs_boxes: return None, None # Should not happen if abs_boxes_scaled was valid

        return composite_img, final_abs_boxes
    except Exception as e:
        # print(f"Error in composite_panel_realistically: {e}")
        return None, None

In [104]:
def get_initial_bbox(img):
    return [[CLASS_ID, 0.5, 0.5, 1.0, 1.0]]

def denormalize_boxes(boxes, width, height):
    abs_boxes = []
    if width <=0 or height <=0: return []
    for cls, xc, yc, w, h in boxes:
        abs_boxes.append([ cls, int((xc - w / 2) * width), int((yc - h / 2) * height), int((xc + w / 2) * width), int((yc + w / 2) * height)])
    return abs_boxes

def normalize_boxes(boxes, width, height):
    norm_boxes = []
    if width <= 0 or height <= 0: return []
    for cls, x1, y1, x2, y2 in boxes:
        if x2 <= x1 or y2 <= y1: continue # Basic check
        # Clip coordinates
        x1_c = max(0, min(x1, width))
        y1_c = max(0, min(y1, height))
        x2_c = max(0, min(x2, width))
        y2_c = max(0, min(y2, height))
        # Check again after clipping
        if x2_c <= x1_c or y2_c <= y1_c: continue
        # Calculate normalized values
        xc = (x1_c + x2_c) / 2 / width
        yc = (y1_c + y2_c) / 2 / height
        w = (x2_c - x1_c) / width
        h = (y2_c - y1_c) / height
        # Final check for valid width/height
        if w > 1e-6 and h > 1e-6: # Use a small threshold
            norm_boxes.append([cls, xc, yc, w, h])
    return norm_boxes

def save_yolo_labels(label_path, boxes):
    try:
        with open(label_path, 'w') as f:
            for box in boxes:
                f.write(f"{int(box[0])} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    except Exception as e:
        print(f"Error saving label file {label_path}: {e}")

In [105]:
def is_overlapping(boxA, boxB):
    """Sprawdza, czy dwa prostokąty (BBoxy) nachodzą na siebie."""
    if (boxA[0] >= boxB[2]) or (boxA[2] <= boxB[0]):
        return False
    if (boxA[1] >= boxB[3]) or (boxA[3] <= boxB[1]):
        return False
    return True

def find_placement_position(mask, panel_w, panel_h, placed_boxes):
    """
    Próbuje znaleźć losową, prawidłową pozycję dla panelu na danej masce.
    """
    bg_h, bg_w = mask.shape
    allowed_y_coords, allowed_x_coords = np.where(mask == 255)
    
    if len(allowed_x_coords) == 0:
        return None, None, None # Nie ma gdzie umieścić panelu na TEJ masce

    for _ in range(MAX_PLACEMENT_ATTEMPTS):
        idx = random.randint(0, len(allowed_x_coords) - 1)
        center_x, center_y = allowed_x_coords[idx], allowed_y_coords[idx]
        
        x_offset = int(center_x - panel_w / 2)
        y_offset = int(center_y - panel_h / 2)
        
        if x_offset < 0 or y_offset < 0 or (x_offset + panel_w) > bg_w or (y_offset + panel_h) > bg_h:
            continue 

        if y_offset > (bg_h * POSITION_MAX_Y):
             continue 

        new_box = [x_offset, y_offset, x_offset + panel_w, y_offset + panel_h]
        
        has_collision = False
        for existing_box in placed_boxes:
            if is_overlapping(new_box, existing_box):
                has_collision = True
                break
        
        if not has_collision:
            return x_offset, y_offset, new_box
            
    return None, None, None # Nie udało się znaleźć miejsca po X próbach

Pętla główna

In [ ]:
print("Rozpoczynam generowanie danych...")
start_time_total = time.time()

roof_files = [f for f in os.listdir(ROOF_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
if not roof_files: print(f"BŁĄD KRYTYCZNY: Brak plików w katalogu dachów: {ROOF_DIR}.")
else: print(f"Znaleziono {len(roof_files)} obrazów tła.")

panel_files = [f for f in os.listdir(PANEL_DIR) if f.lower().endswith('.png')]
if not panel_files: print(f"BŁĄD KRYTYCZNY: Brak plików .png w katalogu paneli: {PANEL_DIR}.")
else: print(f"Znaleziono {len(panel_files)} plików paneli. Rozpoczynam pętlę główną...")

total_generated_count = 0
attempt_count = 0 
max_total_attempts = TOTAL_IMAGES_TO_GENERATE * 10 

while total_generated_count < TOTAL_IMAGES_TO_GENERATE and attempt_count < max_total_attempts:
    attempt_count += 1
    if not roof_files or not panel_files:
        print("Brak plików tła lub paneli. Zatrzymuję generowanie.")
        break
        
    try:
        # 1. Wczytaj tło i stwórz maskę
        roof_file = random.choice(roof_files)
        roof_path = os.path.join(ROOF_DIR, roof_file)
        try: roof_img = Image.open(roof_path).convert("RGB")
        except Exception as e_open_roof: continue
        
        try: resample_filter = Image.Resampling.LANCZOS
        except AttributeError: resample_filter = Image.LANCZOS
        roof_img = roof_img.resize(FINAL_IMAGE_SIZE, resample_filter) 

        # Konwersja do HSV dla maski
        roof_hsv = cv2.cvtColor(np.array(roof_img), cv2.COLOR_RGB2HSV)
        
        # ZMIANA: Stwórz maski dla wszystkich priorytetów
        mask_p1 = create_priority_mask(roof_hsv, priority_level=1)
        mask_p2 = create_priority_mask(roof_hsv, priority_level=2)
        mask_p3 = create_priority_mask(roof_hsv, priority_level=3)
        
        # Przygotuj obraz do wklejania
        composite_img = roof_img.copy()
        placed_boxes_on_this_roof = []
        all_labels_for_this_roof = []
        
        # 2. Pętla umieszczania paneli
        num_panels_to_place = random.randint(MIN_PANELS_PER_IMAGE, MAX_PANELS_PER_IMAGE)
        
        for _ in range(num_panels_to_place):
            panel_file = random.choice(panel_files)
            panel_path = os.path.join(PANEL_DIR, panel_file)
            try: original_panel_img = Image.open(panel_path).convert("RGBA")
            except Exception as e: continue 
            
            initial_boxes_norm = get_initial_bbox(original_panel_img)
            
            panel_copy = original_panel_img.copy()
            abs_boxes = denormalize_boxes(initial_boxes_norm, panel_copy.width, panel_copy.height)
            
            # Główna funkcja transformacji (z rozmyciem i kontrolą widoczności)
            aug_panel, aug_abs_boxes = transform_image_and_boxes(panel_copy, abs_boxes)
            
            if aug_panel is None or not aug_abs_boxes:
                continue # Augmentacja nie powiodła się lub panel stał się niewidoczny

            # --- Skalowanie panelu ---
            target_fill_percent = random.uniform(SCALE_RANGE[0], SCALE_RANGE[1])
            target_panel_w = int(FINAL_IMAGE_SIZE[0] * target_fill_percent)
            scale_needed = target_panel_w / aug_panel.size[0] if aug_panel.size[0] > 0 else 0
            if scale_needed <= 0: continue

            new_w = max(MIN_PIXEL_SIZE, target_panel_w)
            new_h = max(MIN_PIXEL_SIZE, int(aug_panel.size[1] * scale_needed))

            aug_panel = aug_panel.resize((new_w, new_h), resample_filter)
            
            # --- ZMIANA: Logika Priorytetowa Znajdowania Miejsca ---
            
            x_offset, y_offset, new_abs_box = None, None, None

            # 1. Spróbuj P1 (Czerwienie, Pomarańcze)
            x_offset, y_offset, new_abs_box = find_placement_position(
                mask_p1, new_w, new_h, placed_boxes_on_this_roof
            )

            # 2. Jeśli P1 się nie udało, spróbuj P2 (Granat, Czarny)
            if x_offset is None:
                x_offset, y_offset, new_abs_box = find_placement_position(
                    mask_p2, new_w, new_h, placed_boxes_on_this_roof
                )

            # 3. Jeśli P1 i P2 się nie udały, spróbuj P3 (Ciemnoszary)
            if x_offset is None:
                x_offset, y_offset, new_abs_box = find_placement_position(
                    mask_p3, new_w, new_h, placed_boxes_on_this_roof
                )
            
            # --- Koniec zmiany ---
            
            if x_offset is not None:
                # Miejsce znalezione!
                composite_img.paste(aug_panel, (x_offset, y_offset), aug_panel)
                placed_boxes_on_this_roof.append(new_abs_box)
                
                # Przelicz oryginalny BBox panelu na finalne miejsce
                for cls, x1, y1, x2, y2 in aug_abs_boxes:
                    x1_s = int(x1 * scale_needed) + x_offset
                    y1_s = int(y1 * scale_needed) + y_offset
                    x2_s = int(x2 * scale_needed) + x_offset
                    y2_s = int(y2 * scale_needed) + y_offset
                    if x2_s > x1_s and y2_s > y1_s:
                        all_labels_for_this_roof.append([cls, x1_s, y1_s, x2_s, y2_s])

        # 3. Zapisz obraz (jeśli umieszczono jakiekolwiek panele)
        if all_labels_for_this_roof:
            norm_boxes = normalize_boxes(all_labels_for_this_roof, FINAL_IMAGE_SIZE[0], FINAL_IMAGE_SIZE[1])
            
            if norm_boxes: 
                out_name = f"roof_comp_4_{total_generated_count:04d}"
                img_save_path = os.path.join(OUTPUT_IMG_DIR, out_name + ".jpg")
                label_save_path = os.path.join(OUTPUT_LABEL_DIR, out_name + ".txt")

                composite_img.save(img_save_path)
                save_yolo_labels(label_save_path, norm_boxes)

                total_generated_count += 1

                if total_generated_count % 10 == 0:
                     print(f"    ...Wygenerowano {total_generated_count}/{TOTAL_IMAGES_TO_GENERATE} obrazów (próba {attempt_count})")

    except Exception as e:
        print(f"BŁĄD KRYTYCZNY (w głównej pętli): {e}. Próbuję dalej...")
        import traceback
        traceback.print_exc() 
        continue

total_time = time.time() - start_time_total
if total_generated_count < TOTAL_IMAGES_TO_GENERATE:
     print(f"⚠️ Generowanie zakończono przedwcześnie po {attempt_count} próbach.")
else:
     print(f"✅ Generowanie zakończone.")
     
     
print(f"Łącznie wygenerowano {total_generated_count} obrazów w {total_time:.2f}s.")
print(f"Zbiór danych znajduje się w '{OUTPUT_BASE_DIR}'.")

Rozpoczynam generowanie danych...
Znaleziono 48 obrazów tła.
Znaleziono 5 plików paneli. Rozpoczynam pętlę główną...
    ...Wygenerowano 10/250 obrazów (próba 19)
    ...Wygenerowano 20/250 obrazów (próba 38)
    ...Wygenerowano 30/250 obrazów (próba 57)
    ...Wygenerowano 40/250 obrazów (próba 79)
    ...Wygenerowano 50/250 obrazów (próba 100)
    ...Wygenerowano 60/250 obrazów (próba 119)
    ...Wygenerowano 70/250 obrazów (próba 145)
    ...Wygenerowano 80/250 obrazów (próba 159)
    ...Wygenerowano 90/250 obrazów (próba 180)
    ...Wygenerowano 100/250 obrazów (próba 198)
    ...Wygenerowano 110/250 obrazów (próba 221)
    ...Wygenerowano 120/250 obrazów (próba 241)
    ...Wygenerowano 130/250 obrazów (próba 262)
    ...Wygenerowano 140/250 obrazów (próba 278)
    ...Wygenerowano 150/250 obrazów (próba 300)
    ...Wygenerowano 160/250 obrazów (próba 326)
    ...Wygenerowano 170/250 obrazów (próba 352)
    ...Wygenerowano 180/250 obrazów (próba 371)
    ...Wygenerowano 190/250 obra